# Build a BPE tokenizer from scratch (optional)
**LLMs From Basics · Lesson P3.3**

You'll implement byte-pair encoding in plain Python: count pairs, merge, train, encode and decode, then switch to **bytes** so that any text (emoji included) round-trips.

Each exercise has a cell with a `# TODO`, followed by a **check** cell with `assert`s. Run it; silence plus a ✓ means success.

Stuck? Solutions are at the very bottom. Try first: *struggling to recall is what makes it stick.*

No libraries beyond the Python standard library and `matplotlib` (pre-installed on Colab).

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt

# The tiny corpus from the lesson's animation: word -> how often it appears
toy = {"low": 4, "lowest": 7, "fastest": 5, "longest": 5}
print("ready")

## 1 · Count adjacent pairs
Represent the corpus as a dict mapping each word, **as a tuple of symbols**, to its count, e.g. `('l','o','w'): 4`.

Write `get_pair_counts(word_freqs)` that returns a `Counter` mapping every adjacent pair `(a, b)` to its count, **weighted by the word's frequency**. A pair that occurs twice in a word counts twice.

In [ ]:
def get_pair_counts(word_freqs):
    pairs = Counter()
    # TODO: for each (symbols, count), add `count` for every adjacent pair in `symbols`
    raise NotImplementedError("Exercise 1")
    return pairs

**Check 1** · these are the numbers from the animation.

In [ ]:
wf = {tuple(w): c for w, c in toy.items()}
pc = get_pair_counts(wf)
assert pc[("s", "t")] == 22, pc[("s", "t")]
assert pc[("e", "s")] == 17 and pc[("l", "o")] == 16 and pc[("o", "w")] == 11
assert pc.most_common(1)[0][0] == ("s", "t")
print("✓ exercise 1:", pc.most_common(3))

## 2 · Merge one pair
Write `merge_pair(symbols, pair)`: scan **left to right** and replace each non-overlapping occurrence of `pair = (a, b)` with the single symbol `a + b`.

It must work for strings *and* for `bytes` (we'll need that in exercise 5): `a + b` concatenates both.

In [ ]:
def merge_pair(symbols, pair):
    out = []
    # TODO: walk through `symbols`; when symbols[i], symbols[i+1] == pair, append the merged symbol and skip ahead
    raise NotImplementedError("Exercise 2")
    return out

**Check 2**

In [ ]:
assert merge_pair(list("fastest"), ("s", "t")) == ["f", "a", "st", "e", "st"]
assert merge_pair(["a", "a", "a"], ("a", "a")) == ["aa", "a"]          # left to right, no overlaps
assert merge_pair(["lo", "w", "est"], ("lo", "w")) == ["low", "est"]
assert merge_pair([b"h", b"i"], (b"h", b"i")) == [b"hi"]
print("✓ exercise 2")

## 3 · Train BPE
Put it together. `train_bpe(word_counts, num_merges)` takes `{word: count}` (words as strings), splits each word into characters, then repeats `num_merges` times:
1. count pairs, 2. pick the most frequent, 3. merge it in every word, 4. record it.

Return the list of merges, in order. Stop early if no pair is left.

In [ ]:
def train_bpe(word_counts, num_merges):
    word_freqs = {tuple(w): c for w, c in word_counts.items()}
    merges = []
    # TODO: loop num_merges times: count, pick best (Counter.most_common), merge everywhere, record
    raise NotImplementedError("Exercise 3")
    return merges

**Check 3** · the five merges from the animation.

In [ ]:
merges = train_bpe(toy, 5)
assert merges == [("s", "t"), ("e", "st"), ("l", "o"), ("lo", "w"), ("low", "est")], merges
print("✓ exercise 3:", [a + b for a, b in merges])

## 4 · Encode and decode
To **encode** a new word: split it into characters, then apply the merges **in the order they were learned**. To **decode**: just join the tokens.

The lesson claims the unseen word *slowest* becomes `s + lowest`. Check it.

In [ ]:
def encode_word(word, merges):
    symbols = list(word)
    # TODO: apply each merge in order with merge_pair
    raise NotImplementedError("Exercise 4a")

def decode(tokens):
    # TODO: turn a list of tokens back into a string
    raise NotImplementedError("Exercise 4b")

**Check 4**

In [ ]:
assert encode_word("slowest", merges) == ["s", "lowest"]
assert encode_word("lost", merges) == ["lo", "st"]
assert encode_word("fastest", merges) == ["f", "a", "st", "est"]
for w in ["slowest", "longer", "west", "fog"]:
    assert decode(encode_word(w, merges)) == w
print("✓ exercise 4")

## 5 · Byte-level BPE
Character-level BPE can't encode a character it never saw. GPT-2's fix: start from the **256 possible bytes** of the UTF-8 encoding.

Below, a small multilingual corpus is **pre-tokenized** with a regular expression (a simplified version of what GPT tokenizers do: words keep their leading space), each chunk becomes a tuple of single-byte `bytes` objects, and we train with *your* `get_pair_counts` and `merge_pair`. The vocabulary maps every token (a `bytes` object) to an integer ID.

Your job: write `decode_ids(ids)`, which turns IDs back into a string. Join the bytes **first**, then decode UTF-8 once (a single token may hold only part of a multi-byte character, so decoding token by token can fail).

In [ ]:
text = (
    "the tokenizer reads the text and splits it into tokens. tokens are the pieces a model reads. "
    "common words become single tokens, while rare words are split into smaller pieces. "
    "the model never reads letters, only tokens. a tokenizer that learns more merges makes longer tokens, "
    "so the same text needs fewer tokens. fewer tokens means faster reading, but a bigger vocabulary to learn. "
    "café, naïve, Здравствуйте, こんにちは 🙂"
)
PAT = re.compile(r" ?\w+| ?[^\w\s]+|\s+")

def pretokenize(s):
    return PAT.findall(s)

def train_bytes(text, num_merges):
    chunks = Counter(pretokenize(text))
    wf = {tuple(bytes([b]) for b in ch.encode("utf-8")): c for ch, c in chunks.items()}
    merges = []
    for _ in range(num_merges):
        pc = get_pair_counts(wf)
        if not pc:
            break
        best = pc.most_common(1)[0][0]
        wf = {tuple(merge_pair(list(s), best)): c for s, c in wf.items()}
        merges.append(best)
    vocab = {bytes([b]): b for b in range(256)}
    for a, b in merges:
        vocab[a + b] = len(vocab)
    return merges, vocab

def encode_ids(s, merges, vocab):
    ids = []
    for ch in pretokenize(s):
        syms = [bytes([b]) for b in ch.encode("utf-8")]
        for m in merges:
            syms = merge_pair(syms, m)
        ids += [vocab[t] for t in syms]
    return ids

bmerges, vocab = train_bytes(text, 60)
inv_vocab = {i: t for t, i in vocab.items()}
print(len(vocab), "tokens in the vocabulary;", [inv_vocab[i] for i in encode_ids(" the tokens", bmerges, vocab)])

In [ ]:
def decode_ids(ids):
    # TODO: look up each id in inv_vocab, join the bytes, then decode as UTF-8
    raise NotImplementedError("Exercise 5")

**Check 5** · round trips, including text the tokenizer never saw.

In [ ]:
for s in ["the tokens", "naïve café", "Здравствуйте 🙂", "never seen: 𝛑 ≠ 3, 你好!"]:
    ids = encode_ids(s, bmerges, vocab)
    assert decode_ids(ids) == s, s
    assert all(0 <= i < len(vocab) for i in ids)
assert len(encode_ids(" tokens", bmerges, vocab)) < len(" tokens".encode("utf-8")), "merges should shorten common words"
print("✓ exercise 5: every string round-trips; nothing is ever unknown")

## Experiment · vocabulary size vs sequence length
More merges → longer tokens → fewer tokens for the same text. Run this, then try:
- Encode a sentence in a language that is barely in the corpus. How many tokens per character does it need compared with English?
- How does the curve flatten? Why do the last merges save so little?

In [ ]:
sizes, lengths = [], []
for n in [0, 10, 25, 50, 100, 200]:
    m, v = train_bytes(text, n)
    sizes.append(len(v)); lengths.append(len(encode_ids(text, m, v)))
plt.plot(sizes, lengths, "o-")
plt.xlabel("vocabulary size (256 bytes + merges)"); plt.ylabel("tokens needed for the corpus")
plt.title("Bigger vocabulary, shorter sequences"); plt.show()
print(list(zip(sizes, lengths)))

## Recap
- BPE = count adjacent pairs, merge the most frequent, repeat. The output is an **ordered list of merges**.
- Encoding replays the merges in order, so unseen words still split into known pieces.
- Byte-level BPE starts from 256 bytes, so **any** string can be encoded; decode by joining bytes first.
- More merges give shorter sequences, with diminishing returns.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def get_pair_counts(word_freqs):
    pairs = Counter()
    for symbols, count in word_freqs.items():
        for a, b in zip(symbols, symbols[1:]):
            pairs[(a, b)] += count
    return pairs

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def merge_pair(symbols, pair):
    out = []
    i = 0
    while i < len(symbols):
        if i + 1 < len(symbols) and (symbols[i], symbols[i + 1]) == pair:
            out.append(symbols[i] + symbols[i + 1])
            i += 2
        else:
            out.append(symbols[i])
            i += 1
    return out

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def train_bpe(word_counts, num_merges):
    word_freqs = {tuple(w): c for w, c in word_counts.items()}
    merges = []
    for _ in range(num_merges):
        pairs = get_pair_counts(word_freqs)
        if not pairs:
            break
        best = pairs.most_common(1)[0][0]
        word_freqs = {tuple(merge_pair(list(s), best)): c for s, c in word_freqs.items()}
        merges.append(best)
    return merges

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def encode_word(word, merges):
    symbols = list(word)
    for m in merges:
        symbols = merge_pair(symbols, m)
    return symbols

def decode(tokens):
    return "".join(tokens)

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def decode_ids(ids):
    return b"".join(inv_vocab[i] for i in ids).decode("utf-8")